# Exoplanet Habitability

Goal: use the NASA Exoplanet Archive's Planetary Systems Composite Parameters table (`pscomppars`) to find rocky exoplanets in the habitable zone, then compare our list with NASA-linked research (Kopparapu et al. 2014, the HWO target stars, and Bohl et al. 2026).

## Setup

This cell finds the project folder and adds it to Python's search path, so we can import our own tested code with `from src import physics`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The notebook usually runs from inside notebooks/, so the project root is one level up
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import physics

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

# Create any missing folders (does nothing if they already exist)
for folder in [RAW_DIR, PROCESSED_DIR, FIGURES_DIR, OUTPUTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("pandas", pd.__version__, "| numpy", np.__version__)

### pandas 2 vs pandas 3: what changes for us

This notebook runs on pandas 2.x (what Google Colab has) and pandas 3.x (local). The setup cell above prints which version you are using. Two differences matter here:

**1. Copy-on-Write (the default in pandas 3).** A filtered DataFrame behaves like its own independent copy. So chained assignment such as
`df[df["pl_rade"] > 0]["label"] = "x"` silently does nothing in pandas 3 (pandas 2 only warns with `SettingWithCopyWarning`).
*Our rule:* always assign with `df.loc[mask, "col"] = value`, and call `.copy()` when we keep a filtered subset that we will modify later.

**2. New default string dtype.** Text columns such as `pl_name` show up as `str` in pandas 3 but `object` in pandas 2, so `df.dtypes` looks different in Colab. The data is the same.
*Our rule:* never select text columns with `select_dtypes(include="object")` (in pandas 3 it misses them). Use `pd.api.types.is_string_dtype(...)` or list the columns by name.

When a later output differs between versions, a note there points back to this cell.

## Phase 1: Load the data

### How we get the data: TAP and ADQL

**TAP** (Table Access Protocol) is a standard web interface that astronomy archives use to answer database queries. **ADQL** is the SQL-like language we send to it, for example `SELECT pl_name, pl_rade FROM pscomppars`. We send the query as a web request with `requests` and the archive sends back a CSV file.

### Why the Composite Parameters table (`pscomppars`)?

The default Planetary Systems table (`ps`) keeps **every published solution** for every planet, so one planet can appear many times. When we checked, `ps` had 40,350 rows, and TRAPPIST-1 e alone had 5 rows. You would have to pick one solution per planet yourself. `pscomppars` has **one row per planet**, filled with the best available value for each column, which is what a planet-by-planet analysis needs.

**Caveat from the archive:** the values in one row can come from different papers, so a row is *"not necessarily self-consistent"*. Some values are also **calculated** by the archive when a measurement is missing. For example, it estimates a radius from the mass using the Chen & Kipping (2017) relation. That's why we also keep three source columns: `pl_rade_reflink`, `pl_bmassprov` and `pl_insol_reflink`. ([column docs](https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html), [calculated values](https://exoplanetarchive.ipac.caltech.edu/docs/pscp_calc.html))

In [ ]:
from datetime import date

from src import archive

COLUMNS = [
    "pl_name", "hostname", "discoverymethod", "disc_year",
    "pl_rade", "pl_bmasse", "pl_orbper", "pl_orbsmax", "pl_orbeccen",
    "pl_insol", "pl_eqt",
    "st_teff", "st_rad", "st_lum", "st_mass", "sy_dist",
    # Source columns: say whether a value was measured or calculated by the archive
    "pl_rade_reflink", "pl_bmassprov", "pl_insol_reflink",
]

adql = "SELECT " + ", ".join(COLUMNS) + " FROM pscomppars"
print(adql)

**Load or download.** If a snapshot already exists in `data/raw/`, we load the newest one and do **not** call the API. Set `REFRESH = True` to take a new dated snapshot; old snapshots are never overwritten.

In [ ]:
REFRESH = False  # set to True to take a new dated snapshot

todays_file = RAW_DIR / f"pscomppars_{date.today().isoformat()}.csv"
snapshot = archive.latest_snapshot(RAW_DIR, "pscomppars")

if snapshot is None or (REFRESH and not todays_file.exists()):
    print("Downloading a new snapshot from the archive...")
    snapshot = archive.download_tap_csv(adql, todays_file)
else:
    print("Loading existing snapshot (no API call)")

print("File:", snapshot.relative_to(PROJECT_ROOT).as_posix())
planets = pd.read_csv(snapshot)

# If the archive returned an error message instead of data, the header would not match
assert list(planets.columns) == COLUMNS, f"Unexpected columns: {list(planets.columns)}"

print("Shape (rows, columns):", planets.shape)
print(f"Planets in table: {len(planets):,}")

**Data types.** Numbers are `float64` or `int64`. Text columns show as `str` on pandas 3 and `object` on pandas 2; the data is the same (see the *pandas 2 vs pandas 3* cell).

In [ ]:
planets.dtypes.to_frame("dtype")

**Summary statistics.** `describe()` gives the count, mean, spread and percentiles of each numeric column. We transpose it (`.T`) so each column becomes a row, which is easier to read.

In [ ]:
planets.describe().T

**Percent missing per column.** For each column, the share of planets with no value. This tells us which later steps will lose planets.

In [ ]:
percent_missing = (
    planets.isna().mean()   # fraction of rows that are missing, per column
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
    .to_frame("percent_missing")
)
percent_missing

**Measured or calculated?** The reference column says "Calculated Value" when the archive computed the number itself instead of taking it from a paper. `pl_bmassprov` tells us whether the mass is a true mass, a minimum mass (`Msini`, from radial velocity), or estimated from the radius (`M-R relationship`).

In [ ]:
radius_calculated = planets["pl_rade_reflink"].str.contains("Calculated Value", na=False)
insol_calculated = planets["pl_insol_reflink"].str.contains("Calculated Value", na=False)

print(f"Radii calculated from mass by the archive: {radius_calculated.sum():,} "
      f"of {planets['pl_rade'].notna().sum():,} planets with a radius")
print(f"Insolations calculated by the archive:     {insol_calculated.sum():,} "
      f"of {planets['pl_insol'].notna().sum():,} planets with an insolation")
print()
print("Where does the mass come from? (pl_bmassprov)")
print(planets["pl_bmassprov"].value_counts(dropna=False).to_string())

In [ ]:
# Calculated radii by discovery method: radial velocity planets almost never have a measured radius
pd.crosstab(planets["discoverymethod"], radius_calculated.rename("radius_calculated"))

### What stands out (snapshot of 2026-10-24)

- **6,451 planets**, one row each. Radius is missing for only 0.8%, but eccentricity is missing for 16.6% and insolation for 9.2%. Dropping every row with *any* missing value would throw away far more planets than any single step needs (Phase 2).
- **Many radii are not measured.** 1,539 of 6,401 radii were calculated by the archive from the mass. That covers 1,134 of the radial velocity planets and all 292 microlensing planets, but only 2 transit planets. A "rocky" label based on these radii is really a mass-based guess.
- **Masses are often estimates too.** 3,035 masses come from the mass-radius relationship (mostly transit planets with no measured mass), and 924 are only minimum masses (`Msini`).
- **Odd values to handle in Phase 2.** 46 planets have `pl_insol` = 0. These are wide-orbit imaged planets whose tiny flux rounds to zero, not true zeros; we'll recompute S in Phase 3. 3,156 of 5,383 eccentricities are exactly 0, which usually means "assumed circular", not "measured circular". The hottest host (57,000 K) is a white dwarf in the NN Ser system.
- **Huge ranges** (orbital period from 0.09 days to 400 million days, insolation from 0 to 636,000 times Earth's). That's why we will use log scales for plots and clustering.

## Phase 2: Clean and filter

### Why we don't drop every row with a missing value

`planets.dropna()` would delete a planet if *any* column is empty. For example, it would delete a planet just because its eccentricity is unknown (16.6% of planets), even though deciding whether a planet is rocky doesn't need eccentricity. That throws away good data and can bias the sample toward well-studied systems. Instead, each step only drops planets missing the columns *that step* needs. The tradeoff is that different steps use slightly different sets of planets, so we print how many planets remain after every step.

### Step 1: remove physically impossible values

A radius, mass, orbit, temperature or distance of zero or less can't be real. We replace those values with `NaN` (missing) instead of deleting the planet, so it stays available for steps that don't need that column. Eccentricity must satisfy 0 ≤ e < 1 for a closed orbit. `st_lum` is not checked: it is a log10, so negative values are normal. The 46 zero insolations are wide-orbit planets whose tiny flux rounds to 0; Phase 3 recomputes them.

In [ ]:
from src import rocky

assert planets["pl_name"].is_unique, "pl_name should identify each planet once"

clean, removed = rocky.clean_impossible_values(planets)
print(removed[removed > 0].to_string())
print()
print(f"Planets after cleaning: {len(clean):,} (bad values removed, no rows dropped)")

### Step 2: which evidence do we use for each planet?

**Minimum mass.** The radial velocity method measures how much a star wobbles toward and away from us as a planet pulls on it. If the orbit is tilted, we only see part of the wobble, so we get M·sin(i): a *lower bound* on the true mass, never the mass itself.

Each planet gets a `rocky_path`:
1. **excluded**: microlensing planets. Their `pl_orbsmax` is a projected separation, and 290 of 292 have no insolation, so we can't place them in a habitable zone.
2. **measured radius**: a real radius measurement (`pl_rade_reflink` is not "Calculated Value"). This includes 33 planets found by radial velocity that were later seen transiting.
3. **minimum mass**: no measured radius, but a measured mass. We ignore the archive's estimated radius. `mass_type` records whether the mass is a minimum (`Msini`) or a true mass (`Mass`, `Msin(i)/sin(i)`).
4. **excluded**: no measured radius and no usable mass.

In [ ]:
classified = rocky.assign_rocky_path(clean)

not_microlensing = classified["discoverymethod"] != "Microlensing"
print(f"Planets before path assignment:     {len(classified):,}")
print(f"After excluding microlensing:       {not_microlensing.sum():,}")
usable = classified["rocky_path"] != rocky.PATH_EXCLUDED
print(f"After excluding no usable data:     {usable.sum():,}")
print()
print(classified["rocky_path"].value_counts().to_string())

In [ ]:
# Which discovery methods end up on each path?
pd.crosstab(classified["discoverymethod"], classified["rocky_path"], margins=True)

In [ ]:
on_mass_path = classified["rocky_path"] == rocky.PATH_MINIMUM_MASS
print("Mass type on the minimum mass path:")
print(classified.loc[on_mass_path, "mass_type"].value_counts().to_string())

### Step 3: rocky labels

**The radius valley.** Very few planets have radii between about 1.5 and 2.0 Earth radii (Fulton et al. 2017, AJ 154, 109). The usual explanation: planets just below the gap lost their gas envelopes to their star's radiation and are bare rock, while planets above it kept a puffy hydrogen layer ("mini-Neptunes"). 1.6 R⊕ is a common rocky cutoff because Rogers (2015, arXiv:1407.4457) showed that most planets of that size are already too low-density to be pure rock and iron.

| Path | rocky | possibly rocky | not rocky |
|---|---|---|---|
| measured radius | R ≤ 1.6 R⊕ | 1.6 < R ≤ 2.0 R⊕ | R > 2.0 R⊕ |
| minimum mass | (none) | M ≤ 5 M⊕ | M > 5 M⊕ |

**Judgment calls (value, source, alternative):**
- **1.6 R⊕**: Rogers (2015). Alternative: 1.5 R⊕, the lower edge of the Fulton valley.
- **2.0 R⊕**: Bohl et al. (2026, arXiv:2501.14054), §1: *"we adopt a maximum radius of 2 R⊕"*; §3: *"≤2 R⊕, or ≤5 M⊕ for non-transiting planets"* (both cite Hill et al. 2023). So our "rocky + possibly rocky" matches their rocky set, which keeps the Phase 6 comparison fair. Alternative: 1.96 R⊕, the top of the range Bohl et al. quote.
- **5 M⊕**: Bohl et al. §1: *"For planets without measured radii, we adopt a maximum mass of 5 M⊕"*. Alternative: about 6 M⊕, the Earth-like-composition mass at 1.6 R⊕.
- **Mass-path planets are at most "possibly rocky"**: a minimum mass is only a lower bound, and mass alone doesn't tell us what a planet is made of. Alternative: call M ≤ 5 "rocky", as Bohl et al. do.

In [ ]:
classified = rocky.label_rockiness(classified)

has_label = classified["rocky_label"] != ""
print(pd.crosstab(classified.loc[has_label, "rocky_path"],
                  classified.loc[has_label, "rocky_label"], margins=True).to_string())
print()
small = classified["rocky_label"].isin(["rocky", "possibly rocky"])
print(f"Planets with a rocky label:            {has_label.sum():,}")
print(f"Planets that are rocky or possibly rocky: {small.sum():,}")

### Why we never use a mass computed from the radius (circularity)

For 3,035 planets, the archive did not measure the mass. It *computed* the mass from the radius using the Chen & Kipping (2017) formula. If we then asked "is the mass ≤ 5 M⊕?" to confirm that a planet is rocky, we would just be checking the radius again with extra steps: the formula turns R = 2.0 R⊕ into M ≈ 4.7 M⊕, so the two tests agree **by construction**. Agreement like that is not evidence. The table below shows it: for planets with computed masses the agreement is near 100% and means nothing. For planets with a real measured mass, any disagreement is real information about density (a puffy planet or a dense one).

**Rule used everywhere in this project:** a mass with `pl_bmassprov == "M-R relationship"` is never used to support, confirm, or change a rocky label. Our classifier only uses mass for planets *without* a measured radius, and tests in `tests/test_rocky.py` enforce this.

In [ ]:
radius_path = classified[classified["rocky_path"] == rocky.PATH_MEASURED_RADIUS]
radius_says_small = radius_path["pl_rade"] <= rocky.POSSIBLY_ROCKY_MAX_RADIUS
mass_says_small = radius_path["pl_bmasse"] <= rocky.ROCKY_MAX_MASS

for provenance in ["M-R relationship", "Mass"]:
    group = (radius_path["pl_bmassprov"] == provenance) & radius_path["pl_bmasse"].notna()
    agree = (radius_says_small[group] == mass_says_small[group]).mean() * 100
    print(f"Mass from '{provenance}': {group.sum():,} planets, "
          f"radius and mass tests agree {agree:.1f}% of the time")

**Why only 97%, not 100%, for computed masses?** The two cutoffs are not exactly equivalent under the formula: 2.0 R⊕ becomes 4.66 M⊕, and 5 M⊕ becomes about 2.09 R⊕. The next cell shows that every disagreement is a planet between 2.00 and 2.09 R⊕, the sliver between the two cutoffs. It also recomputes the masses from the radii with the Chen & Kipping (2017) formula, using the coefficients the archive lists ([pscp_calc](https://exoplanetarchive.ipac.caltech.edu/docs/pscp_calc.html): log10 R = −0.0925 + 0.589 · log10 M for 2.04–132 M⊕). The archive's "masses" match the formula almost exactly, so they carry no information the radius doesn't already have.

In [ ]:
computed = radius_path[(radius_path["pl_bmassprov"] == "M-R relationship")
                       & radius_path["pl_bmasse"].notna()]
disagree = (computed["pl_rade"] <= rocky.POSSIBLY_ROCKY_MAX_RADIUS) != (computed["pl_bmasse"] <= rocky.ROCKY_MAX_MASS)
print(f"Disagreements: {disagree.sum()}, all with radius between "
      f"{computed.loc[disagree, 'pl_rade'].min():.2f} and {computed.loc[disagree, 'pl_rade'].max():.2f} R⊕")

# Invert log10(R) = -0.0925 + 0.589 * log10(M) to predict M from R (valid for about 1.23-11.1 R⊕)
middle = computed[computed["pl_rade"].between(1.23, 11.1)]
predicted_mass = 10 ** ((np.log10(middle["pl_rade"]) + 0.0925) / 0.589)
ratio = middle["pl_bmasse"] / predicted_mass
print(f"Archive mass / formula mass for {len(middle):,} planets: "
      f"min {ratio.min():.3f}, median {ratio.median():.3f}, max {ratio.max():.3f}")

In [ ]:
processed_file = PROCESSED_DIR / "planets_classified.csv"
classified.to_csv(processed_file, index=False)
print("Saved", processed_file.relative_to(PROJECT_ROOT).as_posix(), classified.shape)

### Step 4: see the radius valley ourselves

We plot measured radii on a **log scale**: equal steps on the axis mean equal *ratios* (1 → 2 is as wide as 10 → 20), which spreads out the small planets where the valley is. The left panel shows all radii, with the archive's calculated radii drawn as an outline so we can see how estimated values behave. The right panel zooms in on planets with orbital periods under 100 days, the same cut Fulton et al. (2017) used, with finer bins.

In [ ]:
from src import plot_style

plot_style.apply_style()

on_radius_path = classified["rocky_path"] == rocky.PATH_MEASURED_RADIUS
measured_radii = classified.loc[on_radius_path, "pl_rade"]
calculated_radii = classified.loc[rocky.radius_is_calculated(classified), "pl_rade"]
short_period_radii = classified.loc[on_radius_path & (classified["pl_orbper"] < 100), "pl_rade"]

fig, (ax_all, ax_zoom) = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [3, 2]})


def draw_cutoffs(ax):
    """Shade Fulton's valley and draw dashed lines at our two radius cutoffs."""
    ax.axvspan(1.5, 2.0, color=plot_style.SHADE, zorder=0,
               label="Radius valley, 1.5–2.0 R⊕ (Fulton et al. 2017)")
    for cutoff in [rocky.ROCKY_MAX_RADIUS, rocky.POSSIBLY_ROCKY_MAX_RADIUS]:
        ax.axvline(cutoff, color=plot_style.TEXT_SECONDARY, linestyle="--", linewidth=1)


# Left: all radii. Log-spaced bins, each about 9% wider than the one before
bins_all = np.logspace(np.log10(0.3), np.log10(40), 57)
draw_cutoffs(ax_all)
ax_all.hist(measured_radii, bins=bins_all, color=plot_style.SERIES[0], edgecolor=plot_style.SURFACE,
            linewidth=0.8, label=f"Measured radius ({len(measured_radii):,} planets)")
ax_all.hist(calculated_radii, bins=bins_all, histtype="step", color=plot_style.SERIES[1], linewidth=1.6,
            label=f"Calculated from mass by the archive ({len(calculated_radii):,})")
ax_all.set_xscale("log")
ax_all.set_xlim(0.3, 40)
ax_all.set_xticks([0.5, 1, 1.6, 2, 3, 5, 10, 20])
ax_all.set_xticklabels(["0.5", "1", "1.6", "2", "3", "5", "10", "20"])
ax_all.set_xlabel("Planet radius (Earth radii, log scale)")
ax_all.set_ylabel("Number of planets")
ax_all.set_title("All planets with a radius")
ax_all.set_ylim(0, ax_all.get_ylim()[1] * 1.45)  # headroom so the legend sits above the bars
ax_all.legend(loc="upper right")

# Right: zoom on small, short-period planets with finer bins
bins_zoom = np.logspace(np.log10(0.7), np.log10(4.5), 31)
draw_cutoffs(ax_zoom)
ax_zoom.hist(short_period_radii, bins=bins_zoom, color=plot_style.SERIES[0],
             edgecolor=plot_style.SURFACE, linewidth=0.8,
             label=f"Measured radius, period < 100 days ({short_period_radii.between(0.7, 4.5).sum():,} shown)")
ax_zoom.set_xscale("log")
ax_zoom.set_xlim(0.7, 4.5)
ax_zoom.set_xticks([0.8, 1, 1.2, 1.6, 2, 2.5, 3, 4])
ax_zoom.set_xticklabels(["0.8", "1", "1.2", "1.6", "2", "2.5", "3", "4"])
ax_zoom.set_xlabel("Planet radius (Earth radii, log scale)")
ax_zoom.set_ylabel("Number of planets")
ax_zoom.set_title("Zoom: small planets, orbital period < 100 days")
ax_zoom.set_ylim(0, ax_zoom.get_ylim()[1] * 1.3)
ax_zoom.legend(loc="upper left")

fig.suptitle("Exoplanet radii and the radius valley (dashed lines: our 1.6 and 2.0 R⊕ cutoffs)",
             fontsize=14, color=plot_style.TEXT_PRIMARY)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "radius_histogram.png")
plt.show()

### What we learned in Phase 2

- **Cleaning removed values, not planets.** Only `pl_insol` had impossible values: 46 zeros, which are rounding for wide orbits. All 6,451 planets stay in the table.
- **Paths:** 4,862 planets use a measured radius, 1,290 use a mass (877 minimum masses, 413 true masses), and 299 are excluded (292 microlensing, 7 with no usable radius or mass).
- **Labels:** 1,143 rocky, 665 possibly rocky (567 by radius, 98 by mass), 4,344 not rocky, so **1,808 planets are rocky or possibly rocky** and move on to the habitable zone step.
- **Circularity is real, not hypothetical.** The archive's radius-derived masses match the Chen & Kipping formula to within 0.5%, so their 97% "agreement" with the radius is guaranteed. Real measured masses agree only 90% of the time, and that 10% is real information about density.
- **The radius valley is faint in this data.** We see a dip near 1.8–1.9 R⊕, not a clean gap, because the archive mixes radii of very different precision. Estimated radii also create a fake spike at 12–15 R⊕, another reason to keep them out of the analysis.

## Phase 3: Stellar energy

## Phase 4: Visualize

## Phase 5: Clustering

## Phase 6: Compare with NASA's own research